# Lakehouse Federation Lab

This notebook demonstrates Databricks Lakehouse Federation — querying a remote PostgreSQL database (Neon) directly from Databricks SQL, enriching with local Delta tables, materializing data locally, and comparing federated vs. local query plans.

## 1. Federation Setup

Create a connection to the remote Neon PostgreSQL instance and register it as a foreign catalog in Unity Catalog.

In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS neon_pg_connection
TYPE POSTGRESQL
OPTIONS (
  host 'ep-square-waterfall-b1ko9duf-pooler.c-5.eu-central-1.aws.neon.tech',
  port '5432',
  user 'neondb_owner',
  password 'npg_placeholder'
);

In [0]:
%sql
CREATE FOREIGN CATALOG IF NOT EXISTS neon_pg_catalog
USING CONNECTION neon_pg_connection
OPTIONS (database 'neondb');

## 2. Querying Remote Data

Query the federated `movie_box_office` table directly from PostgreSQL through the foreign catalog.

In [0]:
%sql
SELECT * FROM neon_pg_catalog.public.movie_box_office;

## 3. Cross-Source Enrichment Join

Join the federated remote table with a local Delta table (`dim_movies`) to enrich with director, country, and computed profit metrics.

In [0]:
%sql
SELECT 
    f.title,
    d.director,
    d.country,
    f.budget_millions,
    f.revenue_millions,
    (f.revenue_millions - f.budget_millions) AS estimated_profit_millions,
    f.rating AS imdb_rating
FROM neon_pg_catalog.public.movie_box_office f
LEFT JOIN main.lab_data.dim_movies d 
    ON LOWER(TRIM(f.title)) = LOWER(TRIM(d.title));

## 4. Materializing Data Locally

Persist the remote table as a local Delta table for improved query performance and reliability, then replay the enrichment join.

In [0]:
%sql
USE CATALOG main;
USE SCHEMA lab_data;

CREATE OR REPLACE TEMPORARY VIEW ext_box_office AS
SELECT * FROM neon_pg_catalog.public.movie_box_office;


CREATE OR REPLACE TABLE local_movie_box_office AS
SELECT * FROM ext_box_office;

In [0]:
%sql
SELECT 
    f.title,
    d.director,
    d.country,
    f.budget_millions,
    f.revenue_millions,
    (f.revenue_millions - f.budget_millions) AS profit_millions,
    f.rating
FROM ext_box_office f
LEFT JOIN dim_movies d 
    ON LOWER(TRIM(f.title)) = LOWER(TRIM(d.title));

## 5. Query Plan Comparison: Federated vs. Local

Compare `EXPLAIN EXTENDED` output for the same filter query (`rating > 8.0`) against the federated JDBC source and the local Parquet/Delta table to observe filter pushdown and storage differences.

In [0]:
%sql
EXPLAIN EXTENDED
SELECT * FROM ext_box_office WHERE rating > 8.0;

In [0]:
%sql
EXPLAIN EXTENDED
SELECT * FROM local_movie_box_office WHERE rating > 8.0;

## Key Takeaways

* **Federated query (Cell 7):** The `rating > 8.0` filter is pushed down to PostgreSQL via JDBC — only matching rows traverse the network.
* **Local query (Cell 8):** The same filter is applied in Photon against Parquet files in S3, with full optimizer statistics available.
* **Materialization trade-off:** Federation provides real-time access to remote data; materialization offers faster, more reliable queries at the cost of freshness.
* **Enrichment gap:** The `dim_movies` join returned NULLs for `director` and `country`, indicating the lookup table may be empty or titles don't match.